<a href="https://colab.research.google.com/github/EnzoAA004/PFI_MVPTest_Enzo_AImodule/blob/research%2Fpost-e50-axial-level-pairing/69_postE50_pathology_module_freeze.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 69 · Post-E50 — Pathology Module Freeze

Objetivo: cerrar rápidamente el módulo P10.7 SPIDER ya entrenado, sin reentrenar.

- No entrena.
- No abre outer validation, internal test RSNA ni test oficial.
- Verifica checkpoint + SHA + contrato.
- Carga el modelo exacto y hace un forward real.
- Audita las 8 tareas y su estado de despliegue.
- Deja contrato final para Backend/Frontend.
- Mantiene bloqueada la inferencia automática de producto hasta demostrar paridad exacta del preprocessing 2.5D P10.7.

Índice:
- 69-A — Scope e identidad del artifact
- 69-B — Contract/metadata audit
- 69-C — Exact model load + GPU forward smoke
- 69-D — Task/deployment readiness
- 69-E — Final pathology freeze contract


In [4]:
# ============================================================
# 69-A FIX — USE VERIFIED P10.7 CHECKPOINT
# ============================================================

PATHOLOGY_CHECKPOINT_PATH = (
    PFI_DRIVE_ROOT
    / "models"
    / "P10_7_spider_degenerative"
    / "final_internal_test_evaluation"
    / "frozen_p10_7_spider_degenerative_multitask.pt"
)

assert PATHOLOGY_CHECKPOINT_PATH.is_file(), PATHOLOGY_CHECKPOINT_PATH

PATHOLOGY_SHA256 = sha256_file(PATHOLOGY_CHECKPOINT_PATH)

print("=" * 72)
print("69-A — PATHOLOGY ARTIFACT IDENTITY")
print("=" * 72)

print("Checkpoint:", PATHOLOGY_CHECKPOINT_PATH)
print("SHA256:", PATHOLOGY_SHA256)

assert PATHOLOGY_SHA256 == EXPECTED_PATHOLOGY_SHA256, (
    f"SHA mismatch: {PATHOLOGY_SHA256}"
)

print("Artifact bytes:", PATHOLOGY_CHECKPOINT_PATH.stat().st_size)
print("69-A ARTIFACT IDENTITY: PASS")

69-A — PATHOLOGY ARTIFACT IDENTITY
Checkpoint: /content/drive/MyDrive/PFI_MVP/models/P10_7_spider_degenerative/final_internal_test_evaluation/frozen_p10_7_spider_degenerative_multitask.pt
SHA256: 16eccff327e6794b127fe372ecd03ea619a0f69d939b84ae1aa2e904191c6293
Artifact bytes: 22182638
69-A ARTIFACT IDENTITY: PASS


In [5]:

# 69-B — CHECKPOINT CONTRACT + METADATA AUDIT
def _importable(name):
    try:
        m=__import__(name); return getattr(m,'__version__','unknown')
    except ImportError:
        return None

pkgs=[]
if _importable('torch') is None: pkgs.append('torch')
if _importable('timm') is None: pkgs.append('timm')
if pkgs:
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet',*pkgs])

import torch, timm

checkpoint=torch.load(
    PATHOLOGY_CHECKPOINT_PATH,
    map_location='cpu',
    weights_only=False
)
assert isinstance(checkpoint,Mapping)

required={
 'schemaVersion','modelStateDict','trainConfig',
 'taskOrder','categoricalTasks','binaryTasks'
}
missing=sorted(required-set(checkpoint.keys()))
assert not missing,f'Missing keys: {missing}'
assert checkpoint['schemaVersion']==EXPECTED_SCHEMA_VERSION
assert list(checkpoint['taskOrder'])==TASK_ORDER
assert dict(checkpoint['categoricalTasks'])==CATEGORICAL_TASKS
assert list(checkpoint['binaryTasks'])==BINARY_TASKS

cfg=checkpoint['trainConfig']
backbone=str(cfg['backbone'])
dropout=float(cfg['dropout'])
image_size=int(cfg.get('image_size',224))
assert backbone=='efficientnet_b0'
assert image_size==224

state=checkpoint['modelStateDict']
assert isinstance(state,Mapping) and len(state)>0

metadata_audit={
 'stage':'69-B',
 'artifact_sha256':PATHOLOGY_SHA256,
 'schema_version':checkpoint['schemaVersion'],
 'task_order':TASK_ORDER,
 'categorical_tasks':CATEGORICAL_TASKS,
 'binary_tasks':BINARY_TASKS,
 'train_config':{
   'backbone':backbone,
   'dropout':dropout,
   'image_size':image_size,
 },
 'state_dict_entries':int(len(state)),
 'human_review_required_metadata':checkpoint.get('humanReviewRequired'),
 'not_clinical_diagnosis_metadata':checkpoint.get('notClinicalDiagnosis'),
 'checkpoint_keys':sorted(map(str,checkpoint.keys())),
 'training_performed':False,
 'rsna_outer_validation_accessed':False,
 'rsna_internal_test_accessed':False,
 'official_test_accessed':False,
}
META_PATH=METRICS_DIR/'69_B_checkpoint_metadata_audit.json'
META_PATH.write_text(json.dumps(metadata_audit,indent=2,default=str))

print('='*72)
print('69-B — CHECKPOINT CONTRACT AUDIT')
print('='*72)
print('schemaVersion:',checkpoint['schemaVersion'])
print('backbone:',backbone)
print('image_size:',image_size)
print('dropout:',dropout)
print('state_dict entries:',len(state))
print('tasks:',len(TASK_ORDER))
print('humanReviewRequired:',checkpoint.get('humanReviewRequired'))
print('notClinicalDiagnosis:',checkpoint.get('notClinicalDiagnosis'))
print('Metadata SHA256:',sha256_file(META_PATH))
print('69-B CONTRACT: PASS')


69-B — CHECKPOINT CONTRACT AUDIT
schemaVersion: pfi.p10-7-research-export.v1
backbone: efficientnet_b0
image_size: 224
dropout: 0.25
state_dict entries: 381
tasks: 8
humanReviewRequired: True
notClinicalDiagnosis: True
Metadata SHA256: 4a51507f10943c113c95987c5ef9f9e2fa8d08d0eca9a23880a9101382fa9e72
69-B CONTRACT: PASS


In [6]:

# 69-C — EXACT MODEL LOAD + REAL GPU FORWARD SMOKE
import torch.nn as nn

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:',DEVICE)
if torch.cuda.is_available():
    print('GPU:',torch.cuda.get_device_name(0))

class P10_7DiscClassifier(nn.Module):
    def __init__(self,backbone='efficientnet_b0',dropout=0.25,pretrained=False):
        super().__init__()
        self.encoder=timm.create_model(
            backbone,pretrained=pretrained,num_classes=0,
            global_pool='avg',in_chans=3
        )
        feature_dim=int(self.encoder.num_features)
        self.ivd_embedding=nn.Embedding(25,16)
        self.trunk=nn.Sequential(
            nn.Linear(feature_dim*2+2+16,512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(float(dropout)),
            nn.Linear(512,256),
            nn.GELU(),
            nn.Dropout(float(dropout)),
        )
        self.heads=nn.ModuleDict({
            'pfirrmann_grade':nn.Linear(256,5),
            'modic_change':nn.Linear(256,4),
            **{task:nn.Linear(256,1) for task in BINARY_TASKS},
        })
    def forward(self,t1,t2,availability,ivd_index):
        t1f=self.encoder(t1)*availability[:,0:1]
        t2f=self.encoder(t2)*availability[:,1:2]
        ivdf=self.ivd_embedding(ivd_index.clamp(0,24))
        shared=self.trunk(torch.cat([t1f,t2f,availability,ivdf],dim=1))
        return {name:head(shared) for name,head in self.heads.items()}

model=P10_7DiscClassifier(backbone=backbone,dropout=dropout,pretrained=False)
model.load_state_dict(checkpoint['modelStateDict'],strict=True)
model.to(DEVICE).eval()
for p in model.parameters(): p.requires_grad_(False)

torch.manual_seed(2026)
BATCH=5
t1=torch.rand(BATCH,3,image_size,image_size,device=DEVICE)
t2=torch.rand(BATCH,3,image_size,image_size,device=DEVICE)
availability=torch.ones(BATCH,2,device=DEVICE)
ivd_index=torch.arange(0,5,dtype=torch.long,device=DEVICE)

t0=time.time()
with torch.inference_mode():
    outputs=model(t1,t2,availability,ivd_index)
if DEVICE.type=='cuda':
    torch.cuda.synchronize()
elapsed_ms=(time.time()-t0)*1000

assert list(outputs.keys())==TASK_ORDER
probability_audit={}
for task,tensor in outputs.items():
    assert torch.isfinite(tensor).all(),task
    if task=='pfirrmann_grade':
        assert tuple(tensor.shape)==(BATCH,5)
    elif task=='modic_change':
        assert tuple(tensor.shape)==(BATCH,4)
    else:
        assert tuple(tensor.shape)==(BATCH,1)

    if task in CATEGORICAL_TASKS:
        probs=torch.softmax(tensor,dim=1)
        assert torch.allclose(probs.sum(1),torch.ones(BATCH,device=DEVICE),atol=1e-5)
        probability_audit[task]={'kind':'categorical','shape':list(probs.shape)}
    else:
        probs=torch.sigmoid(tensor)
        assert ((probs>=0)&(probs<=1)).all()
        probability_audit[task]={'kind':'binary','shape':list(probs.shape)}

forward_summary={
 'stage':'69-C',
 'artifact_sha256':PATHOLOGY_SHA256,
 'device':str(DEVICE),
 'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
 'batch':BATCH,
 'image_size':image_size,
 'levels':SPINE_LEVELS,
 'task_outputs':probability_audit,
 'elapsed_ms_batch':float(elapsed_ms),
 'strict_state_dict_load':True,
 'real_model_forward':True,
 'input_source':'synthetic_structural_smoke_only',
 'accuracy_claimed':False,
 'training_performed':False,
 'rsna_outer_validation_accessed':False,
 'rsna_internal_test_accessed':False,
 'official_test_accessed':False,
}
FORWARD_PATH=METRICS_DIR/'69_C_real_forward_smoke.json'
FORWARD_PATH.write_text(json.dumps(forward_summary,indent=2))

print('='*72)
print('69-C — REAL MODEL FORWARD SMOKE')
print('='*72)
print('Strict state_dict load: PASS')
print('Real forward: PASS')
print('Batch:',BATCH)
print('Tasks:',len(outputs))
print('Elapsed batch ms:',round(elapsed_ms,2))
for task,tensor in outputs.items():
    print(task,tuple(tensor.shape),'FINITE')
print('Forward summary SHA256:',sha256_file(FORWARD_PATH))
print('69-C FORWARD: PASS')


DEVICE: cpu
69-C — REAL MODEL FORWARD SMOKE
Strict state_dict load: PASS
Real forward: PASS
Batch: 5
Tasks: 8
Elapsed batch ms: 429.57
pfirrmann_grade (5, 5) FINITE
modic_change (5, 4) FINITE
upper_endplate_change (5, 1) FINITE
lower_endplate_change (5, 1) FINITE
spondylolisthesis (5, 1) FINITE
disc_herniation (5, 1) FINITE
disc_narrowing (5, 1) FINITE
disc_bulging (5, 1) FINITE
Forward summary SHA256: 719e2588f937cd1f1d758c030aaf6b35ccecfb7713a4a5358a1e023cb575d72b
69-C FORWARD: PASS


In [7]:

# 69-D — TASK / DEPLOYMENT READINESS
rows=[]
for task in TASK_ORDER:
    kind='categorical' if task in CATEGORICAL_TASKS else 'binary'
    classes=CATEGORICAL_TASKS[task] if task in CATEGORICAL_TASKS else 2
    deployment=DEPLOYMENT_STATUS_BY_TASK[task]
    rows.append({
      'finding_type':task,
      'kind':kind,
      'classes':classes,
      'deployment_status':deployment,
      'checkpoint_verified':True,
      'real_forward_verified':True,
      'human_review_required':True,
      'not_clinical_diagnosis':True,
      'product_auto_enable':deployment=='supported_internal',
    })

task_df=pd.DataFrame(rows)
TASK_CSV=RESULTS_DIR/'69_D_task_readiness.csv'
task_df.to_csv(TASK_CSV,index=False)

supported=task_df[task_df.deployment_status=='supported_internal'].finding_type.tolist()
experimental=task_df[task_df.deployment_status=='experimental'].finding_type.tolist()
not_supported=task_df[task_df.deployment_status=='not_product_supported'].finding_type.tolist()

readiness={
 'stage':'69-D',
 'checkpoint_verified':True,
 'real_forward_verified':True,
 'supported_internal':supported,
 'experimental':experimental,
 'not_product_supported':not_supported,
 'preprocessing_parity_status':'NOT_YET_PROVEN',
 'automatic_level_localization_post_e50':{
   'upstream_stage':'67B2',
   'status':'VALIDATED_WITH_ABSTENTION',
   'note':'67B2 no demuestra por sí solo paridad del crop 2.5D P10.7.'
 },
 'ready_for_internal_preprocessed_crop_inference':True,
 'ready_for_automatic_product_inference':False,
 'training_required_now':False,
}
READINESS_PATH=METRICS_DIR/'69_D_pathology_readiness.json'
READINESS_PATH.write_text(json.dumps(readiness,indent=2))

print('='*72)
print('69-D — PATHOLOGY TASK READINESS')
print('='*72)
print(task_df.to_string(index=False))
print()
print('supported_internal:',supported)
print('experimental:',experimental)
print('not_product_supported:',not_supported)
print('Preprocessing parity:',readiness['preprocessing_parity_status'])
print('READY FOR INTERNAL PREPROCESSED-CROP INFERENCE:',
      readiness['ready_for_internal_preprocessed_crop_inference'])
print('READY FOR AUTOMATIC PRODUCT INFERENCE:',
      readiness['ready_for_automatic_product_inference'])
print('Task CSV SHA256:',sha256_file(TASK_CSV))
print('Readiness SHA256:',sha256_file(READINESS_PATH))


69-D — PATHOLOGY TASK READINESS
         finding_type        kind  classes     deployment_status  checkpoint_verified  real_forward_verified  human_review_required  not_clinical_diagnosis  product_auto_enable
      pfirrmann_grade categorical        5          experimental                 True                   True                   True                    True                False
         modic_change categorical        4 not_product_supported                 True                   True                   True                    True                False
upper_endplate_change      binary        2    supported_internal                 True                   True                   True                    True                 True
lower_endplate_change      binary        2    supported_internal                 True                   True                   True                    True                 True
    spondylolisthesis      binary        2 not_product_supported                 Tr

In [8]:

# 69-E — FINAL PATHOLOGY FREEZE CONTRACT
contract={
 'contract_version':'69.1',
 'module':'disc_degenerative_findings',
 'model_id':'spider_degenerative_multitask_sagittal_t1_t2_2p5d',
 'checkpoint':{
   'filename':PATHOLOGY_CHECKPOINT_PATH.name,
   'sha256':PATHOLOGY_SHA256,
   'schema_version':EXPECTED_SCHEMA_VERSION,
   'backbone':backbone,
   'image_size':image_size,
   'strict_state_dict_load':True,
   'real_forward_verified':True,
 },
 'tasks':{
   task:{
     'deployment_status':DEPLOYMENT_STATUS_BY_TASK[task],
     'human_review_required':True,
   } for task in TASK_ORDER
 },
 'localization':{
   'absolute_disc_level_upstream':'67B2',
   'absolute_disc_level_status':'VALIDATED_WITH_ABSTENTION',
   'p10_7_crop_preprocessing_parity':'NOT_YET_PROVEN',
 },
 'safety':{
   'human_review_required':True,
   'not_clinical_diagnosis':True,
   'autonomous_diagnosis':False,
 },
 'scope':{
   'training_performed':False,
   'rsna_outer_validation_accessed':False,
   'rsna_internal_test_accessed':False,
   'official_test_accessed':False,
 },
 'decision':{
   'retrain_now':False,
   'freeze_existing_checkpoint':True,
   'ready_for_internal_preprocessed_crop_inference':True,
   'ready_for_automatic_product_inference':False,
   'next_blocker':(
      '70 — exact P10.7 sagittal T1/T2 2.5D preprocessing parity '
      'from automatic 67B2 disc localization.'
   ),
 },
}
CONTRACT_PATH=METRICS_DIR/'69_E_pathology_freeze_contract.json'
CONTRACT_PATH.write_text(json.dumps(contract,indent=2))

summary={
 'stage':'69-E',
 'status':'PATHOLOGY_CHECKPOINT_FROZEN_NO_RETRAINING',
 'artifact_sha256':PATHOLOGY_SHA256,
 'checkpoint_contract':'PASS',
 'strict_load':'PASS',
 'real_forward':'PASS',
 'task_count':len(TASK_ORDER),
 'supported_internal_count':len(supported),
 'experimental_count':len(experimental),
 'not_product_supported_count':len(not_supported),
 'training_required_now':False,
 'automatic_product_inference_ready':False,
 'next_stage':'70_preprocessing_parity_and_automatic_disc_crop',
 'contract_sha256':sha256_file(CONTRACT_PATH),
 'metadata_sha256':sha256_file(META_PATH),
 'forward_sha256':sha256_file(FORWARD_PATH),
 'readiness_sha256':sha256_file(READINESS_PATH),
}
SUMMARY_PATH=METRICS_DIR/'69_E_final_summary.json'
SUMMARY_PATH.write_text(json.dumps(summary,indent=2))

print('='*72)
print('69-E — PATHOLOGY MODULE FREEZE COMPLETE')
print('='*72)
print('Status:',summary['status'])
print('Artifact SHA256:',PATHOLOGY_SHA256)
print('Checkpoint contract: PASS')
print('Strict model load: PASS')
print('Real forward: PASS')
print('Tasks:',len(TASK_ORDER))
print('Supported internal:',len(supported))
print('Experimental:',len(experimental))
print('Not product supported:',len(not_supported))
print('TRAINING REQUIRED NOW:',summary['training_required_now'])
print('AUTOMATIC PRODUCT INFERENCE READY:',summary['automatic_product_inference_ready'])
print('Next stage:',summary['next_stage'])
print('Contract SHA256:',sha256_file(CONTRACT_PATH))
print('Final summary SHA256:',sha256_file(SUMMARY_PATH))


69-E — PATHOLOGY MODULE FREEZE COMPLETE
Status: PATHOLOGY_CHECKPOINT_FROZEN_NO_RETRAINING
Artifact SHA256: 16eccff327e6794b127fe372ecd03ea619a0f69d939b84ae1aa2e904191c6293
Checkpoint contract: PASS
Strict model load: PASS
Real forward: PASS
Tasks: 8
Supported internal: 4
Experimental: 1
Not product supported: 3
TRAINING REQUIRED NOW: False
AUTOMATIC PRODUCT INFERENCE READY: False
Next stage: 70_preprocessing_parity_and_automatic_disc_crop
Contract SHA256: 20f6007821272be0dc7b44213fc434cc1253f774e2bd6cec30160f5dc6682624
Final summary SHA256: 46c71da4db1d78f5e7a1704fcbe04e896f032d62a0a437eda27f30e8347dfd56
